# Analysing an Algorand account

Sync one testnet account into a temporary database, then look at its activity with pandas.

In [ ]:
import tempfile
from pathlib import Path

from multichain_indexer import Indexer

ACCOUNT = "JCN7TCSQEG6YHAQ7F6BS73J2UDXI2L7S7WJU35LYMIK22IL264ZRJO55QM"
folder = tempfile.TemporaryDirectory()
indexer = Indexer("algorand", database=Path(folder.name) / "example.db")
indexer.sync(ACCOUNT)

Exports carry the shared fields as columns and the `native` block as JSON.

In [ ]:
import io
import json

import pandas as pd

frame = pd.read_csv(io.StringIO(indexer.export("csv", account=ACCOUNT)))
frame["type"] = frame["native"].map(lambda native: json.loads(native)["tx_type"])
frame["time"] = pd.to_datetime(frame["time"])
frame[["time", "type", "sender", "amount", "fee"]].head()

In [ ]:
frame.groupby("type")["fee"].agg(["count", "sum"])

In [ ]:
indexer.metrics(days=365, account=ACCOUNT).native

In [ ]:
indexer.close()
folder.cleanup()